# Imports

In [ ]:
import os
import math

import numpy as np
import pandas as pd

from openbabel import pybel
pybel.ob.obErrorLog.StopLogging()
from rdkit import Chem
from rdkit import Chem
from rdkit.Chem import SDMolSupplier, SDWriter
from rdkit.Chem import Draw
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display
from IPython.display import Image, display

import goodvibes.GoodVibes as gv
import goodvibes.thermo as thermo
import goodvibes.io as io
import goodvibes.pes as pes
from morfeus import BuriedVolume
import get_properties_functions_m as gp

this code is a derivation of the original code (https://github.com/SigmanGroup/GetProperties) tailored specifically to the NN library

## Atom Labeling 

In [ ]:
# generates sdfs for logs in directory + list of IDs for logs in directory 
gp.get_sdf_from_log()
gp.get_log_ids()

In [ ]:
# go through ligands in directory and see if core has 5/6 atoms and if pyox/not
mols_5, mols_6, ligand_core_class = gp.get_ligand_core_type()
#get initial atom numbering based on substructure matching
ligand_initial_atom_numbering = gp.get_initial_atom_numbers(mols_5, mols_6)
# make sure the same atom is labeled N/C across the conformer ensemble
ligand_consistent_atom_numbering = gp.make_ensemble_atom_numbering_consistent(ligand_initial_atom_numbering)
#make sure atom labels match atom types (N1 is a nitrogen, C1 is a carbon)
gp.verify_atom_numbering(ligand_consistent_atom_numbering)

In [ ]:
# atom labeling for PyOx and PyIm ligands
df_pyox_corrected = gp.find_pyox_ligands_and_renumber(ligand_consistent_atom_numbering)
# atom numbering for all other ligands, combining with PyOx ligand df 
df_final_atom_numbering = gp.renumber_non_pyox_ligands(df_pyox_corrected, prefix='Lig', suffix='_')
df_final_atom_numbering.to_excel('atom_numbering.xlsx') # to save atom numbering for reference

## Collect Properties

In [ ]:
# read in atom label dataframe 
df = pd.read_excel('atom_numbering.xlsx')
df.reset_index(inplace=True, drop=True)

In [ ]:
#---------------GoodVibes Engergies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) as well as other thermodynamic properties
# #inputs: dataframe, temperature
df = gp.get_goodvibes_e(df, 298.15)

#---------------Frontier Orbitals-----------------
#E(HOMO), E(LUMO), mu(chemical potential or negative of molecular electronegativity), eta(hardness/softness), omega(electrophilicity index)
df = gp.get_frontierorbs(df)

#---------------Polarizability--------------------
#Exact polarizability
df = gp.get_polarizability(df)

#---------------Dipole----------------------------
#Total dipole moment magnitude in Debye
df = gp.get_dipole(df)

#---------------SASA------------------------------
#Uses morfeus to calculat sovlent accessible surface area and the volume under the SASA
df = gp.get_SASA(df)

#---------------NBO-------------------------------
#natural charge from NBO
#requires the Gaussian keyword = "pop=nbo7" in the .com file
nbo_list = ["C1", "C2", "N1", "N2"]
df = gp.get_nbo(df, nbo_list) 

#---------------NMR-------------------------------
# isotropic NMR shift
# requires the Gaussian keyword = "nmr=giao" in the .com file
nmr_list = ["C1", "C2", "N1", "N2"]
df = gp.get_nmr(df, nmr_list) 

#---------------Distance--------------------------
#distance between 2 atoms
dist_list_of_lists = [["N1", "Ni"], ["N2", "Ni"], ["N2", "C2"], ["N1", "C1"]]
df = gp.get_distance(df, dist_list_of_lists) 

#---------------Angle-----------------------------
#angle between 3 atoms
angle_list_of_lists = [["N1", "C1", "C2"], ["N2", "C2", "C1"]]
df = gp.get_angles(df, angle_list_of_lists) 

# #--------------Vbur Scan bisphosphines-----------------
# #this is the same as the above MORFEUS Vbur Scan function, except it removes both Cls on the metal center
# #need to update this function to make it more general...
a_list = ['Ni']
df = gp.get_vbur_scan(df, a_list, 2, 4, .5)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ["N1", "N2", "C2", "C1"]
df = gp.get_vbur_scan_no_metal(df, vbur_list, 3, 5, 0.5)
    
#---------------Sterimol morfeus------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values
#NOTE: this is much faster than the corresponding DBSTEP function (recommendation: use as default/if you don't need Sterimol2Vec)
sterimol_list_of_lists = [["N1", "Ni"], ["C1", "N1"], ["N2", "C2"], ["N2", "Ni"], ["Ni", "N1"], ["N1", "C1"], ["C2", "N2"], ["Ni", "N2"]]
df = gp.get_sterimol_morfeus(df, sterimol_list_of_lists) 

#---------------Buried Sterimol-------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values within a given sphere of radius r_buried
#atoms outside the sphere + 0.5 vdW radius are deleted and the Sterimol vectors are calculated
#for more information: https://kjelljorner.github.io/morfeus/sterimol.html
#inputs: dataframe, list of atom pairs, r_buried
sterimol_list_of_lists = [["N1", "Ni"], ["C1", "N1"], ["N2", "C2"], ["N2", "Ni"], ["Ni", "N1"], ["N1", "C1"], ["C2", "N2"], ["Ni", "N2"]]
df = gp.get_buried_sterimol(df, sterimol_list_of_lists, 5.5) 

#---------------Hirshfeld-------------------------
#Hirshfeld charge, CM5 charge, Hirshfeld atom dipole
#requires the Gaussian keyword = "pop=hirshfeld" in the .com file
a_list = ['N1', 'N2', 'C1', 'C2']
df = gp.get_hirshfeld(df, a_list) 

#-------------Vbur quadrants ------
#uses the MORFEUS buried volume function and splits into quadrants and octants
#define: df, center of sphere, excluded atom1, excluded atom2, z-axis atom1, z-axis atom2
df = gp.get_vbur_quadrants_octants(df, a1 = 'Ni', ex1 = "-H1", ex2 = "-H2", z1 = "N1", z2 = "N2", radius=5.0)

# -------------Bidentate ligand bite angle------------------
# this give the same information as the angle function, but the output is cleaner
# define: df, metal atom, donor atom 1, donor atom 2
df = gp.get_bite_angle(df, 'Ni', "N1", "N2")

df = gp.get_visible_volume(df, 'Ni', ['-H1', '-H2'])

pd.options.display.max_columns = None
df.to_excel('properties_raw.xlsx') # save properties for each conformer to excel 

## Get Condensed Properties 

In [ ]:
# read in all conformer df to get Min/Max/LowE/Boltz
df = pd.read_excel('properties_raw.xlsx')

prefix = "Lig" 
suffix = "_"
atom_columns_to_drop = ["C2", "C1", "N1", "Ni", "N2", "-H1", "-H2"]
energy_col_header = "G(T)_spc(Hartree)"

In [ ]:
compound_list = []
    
for index, row in df.iterrows():
    log_file = row['log_name'] 
    prefix_and_compound = log_file.split(str(suffix)) 
    compound = prefix_and_compound[0].split(str(prefix)) 
    compound_list.append(compound[1])

compound_list = list(set(compound_list))
compound_list.sort() 
print(compound_list)

In [ ]:
all_df_master = pd.DataFrame(columns=[])
properties_df_master = pd.DataFrame(columns=[])

for compound in compound_list: 
    base = f"{prefix}{compound}"
    pattern = rf"^{base}(_|$)"       
    valuesdf = df[df["log_name"].str.match(pattern)]
    valuesdf = valuesdf.drop(columns = atom_columns_to_drop)
    valuesdf = valuesdf.reset_index(drop = True)  
   
    #define columns that won't be included in summary properties or are treated differently because they don't make sense to Boltzmann average
    non_boltz_columns = ["G(Hartree)","∆G(Hartree)","∆G(kcal/mol)", "e^(-∆G/RT)","Mole Fraction"] #don't boltzman average columns containing these strings in the column label
    reg_avg_columns = ['CPU_time_total(hours)', 'Wall_time_total(hours)'] #don't boltzmann average these either, we average them in case that is helpful
    gv_extra_columns = ['E_spc (Hartree)', 'H_spc(Hartree)', 'T', 'T*S', 'T*qh_S', 'ZPE(Hartree)', 'qh_G(T)_spc(Hartree)', "G(T)_spc(Hartree)"]
    gv_extra_columns.remove(str(energy_col_header))
    
    #calculate the summary properties based on all conformers (Boltzmann Average, Minimum, Maximum, Boltzmann Weighted Std)
    valuesdf["∆G(Hartree)"] = valuesdf[energy_col_header] - valuesdf[energy_col_header].min()
    #print (valuesdf["∆G(Hartree)"])
    valuesdf["∆G(kcal/mol)"] = valuesdf["∆G(Hartree)"] * 627.5
    valuesdf["e^(-∆G/RT)"] = np.exp((valuesdf["∆G(kcal/mol)"] * -1000) / (1.987204 * 298.15)) #R is in cal/(K*mol)
    valuesdf["Mole Fraction"] = valuesdf["e^(-∆G/RT)"] / valuesdf["e^(-∆G/RT)"].sum()
    values_boltz_row = []
    values_min_row = []
    values_max_row = []
    values_boltz_stdev_row =[]
    values_range_row = []
    values_exclude_columns = []
    
    for column in valuesdf:
        if "log_name" in column:
            values_boltz_row.append("Boltzmann Averages")
            values_min_row.append("Ensemble Minimum")
            values_max_row.append("Ensemble Maximum")
            values_boltz_stdev_row.append("Boltzmann Standard Deviation")
            values_range_row.append("Ensemble Range")
            values_exclude_columns.append(column) #used later to build final dataframe
        elif any(phrase in column for phrase in non_boltz_columns) or any(phrase in column for phrase in gv_extra_columns):
            values_boltz_row.append("")
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        elif any(phrase in column for phrase in reg_avg_columns):
            values_boltz_row.append(valuesdf[column].mean()) #intended to print the average CPU/wall time in the boltz column
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        else:
            valuesdf[column] = pd.to_numeric(valuesdf[column]) #to hopefully solve the error that sometimes occurs where the float(Mole Fraction) cannot be mulitplied by the string(property)
            values_boltz_row.append((valuesdf[column] * valuesdf["Mole Fraction"]).sum())
            values_min_row.append(valuesdf[column].min())
            values_max_row.append(valuesdf[column].max())
            values_range_row.append(valuesdf[column].max() - valuesdf[column].min())

        
            # this section generates the weighted std deviation (weighted by mole fraction) 
            # formula: https://www.statology.org/weighted-standard-deviation-excel/
    
            boltz = (valuesdf[column] * valuesdf["Mole Fraction"]).sum() #number
            #print (boltz)
            delta_values_sq = []
    
            #makes a list of the "deviation" for each conformer           
            for index, row in valuesdf.iterrows(): 
                value = row[column]
                delta_value_sq = (value - boltz)**2
                delta_values_sq.append(delta_value_sq)
            
            #w is list of weights (i.e. mole fractions)
            w = list(valuesdf["Mole Fraction"])
            wstdev = np.sqrt( (np.average(delta_values_sq, weights=w)) / (((len(w)-1)/len(w))*np.sum(w)) )
            if len(w) == 1: #if there is only one conformer in the ensemble, set the weighted standard deviation to 0 
                wstdev = 0
            #np.average(delta_values_sq, weights=w) generates sum of each (delta_value_sq * mole fraction)
            
            values_boltz_stdev_row.append(wstdev)
            #print (wstdev)
            
    valuesdf.loc[len(valuesdf)] = values_boltz_row
    valuesdf.loc[len(valuesdf)] = values_boltz_stdev_row
    valuesdf.loc[len(valuesdf)] = values_min_row
    valuesdf.loc[len(valuesdf)] = values_max_row
    valuesdf.loc[len(valuesdf)] = values_range_row

    #final output format is built here:
    explicit_order_front_columns = ["log_name", energy_col_header,"∆G(Hartree)","∆G(kcal/mol)","e^(-∆G/RT)","Mole Fraction"]
    
    #reorders the dataframe using front columns defined above
    valuesdf = valuesdf[explicit_order_front_columns + [col for col in valuesdf.columns if col not in explicit_order_front_columns and col not in values_exclude_columns]]
    
    #determine the index of the lowest energy conformer
    low_e_index = valuesdf[valuesdf["∆G(Hartree)"] == 0].index.tolist()
    
    #copy the row to a new_row with the name of the log changed to Lowest E Conformer
    new_row = valuesdf.loc[low_e_index[0]]
    new_row['log_name'] = "Lowest E Conformer"   
    valuesdf =  valuesdf.append(new_row, ignore_index=True)
    
    #appends the frame to the master output
    all_df_master = pd.concat([all_df_master, valuesdf])
    
    #drop all the individual conformers
    dropindex = valuesdf[valuesdf["log_name"].str.match(pattern)].index
    valuesdf = valuesdf.drop(dropindex)
    valuesdf = valuesdf.reset_index(drop = True)
    
    #drop the columns created to determine the mole fraction and some that 
    valuesdf = valuesdf.drop(columns = explicit_order_front_columns)
    try:
        valuesdf = valuesdf.drop(columns = gv_extra_columns)
    except:
        pass
    try:
        valuesdf = valuesdf.drop(columns = reg_avg_columns)
    except:
        pass
        
#---------------------THIS MAY NEED TO CHANGE DEPENDING ON HOW YOU LABEL YOUR COMPOUNDS------------------------------  
    compound_name = prefix + str(compound) 
#--------------------------------------------------------------------------------------------------------------------      

    properties_df = pd.DataFrame({'Compound_Name': [compound_name]})
   
    for (columnName, columnData) in valuesdf.iteritems():
        properties_df[str(columnName) + "_Boltz"] = [columnData.values[0]]
        properties_df[str(columnName) + "_Boltz_stdev"] = [columnData.values[1]]
        properties_df[str(columnName) + "_min"] = [columnData.values[2]]
        properties_df[str(columnName) + "_max"] = [columnData.values[3]]
        properties_df[str(columnName) + "_range"] = [columnData.values[4]]
        properties_df[str(columnName) + "_low_E"] = [columnData.values[5]]
        
    properties_df_master = pd.concat([properties_df_master, properties_df], axis = 0)

all_df_master = all_df_master.reset_index(drop = True)
properties_df_master = properties_df_master.reset_index(drop = True)

In [ ]:
suffixes_to_drop = ['_stdev', '_range']
pattern = '|'.join(suffixes_to_drop)

columns_to_drop = properties_df_master.filter(regex=pattern).columns
summary_properties = properties_df_master.drop(columns_to_drop, axis=1)

summary_properties.to_excel ('Summary_condensed_properties_hydride_ligands.xlsx')
all_df_master.to_excel('All_Conformer_Properties_hydride_ligands.xlsx', index = False)